<a href="https://colab.research.google.com/github/dkalchenko01/UA-Locations-NER-Transformers/blob/roberta-large/notebooks/03_xlm_roberta_large.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XLM-RoBERTa-large with Weighted Layer Pooling

- Pretrained multilingual Transformer encoder
- Learnable weighted pooling of the last 4 hidden layers
- Token-level BIO classification
- Reannoatated dataset

## Environment Setup

In [1]:
%pip install -q transformers datasets accelerate sentencepiece seqeval wandb kagglehub

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import os
import sys
import ast
import json
import random
import gc
import shutil
import platform
import subprocess
import importlib
from pathlib import Path
from collections import Counter
from importlib.metadata import version
from time import perf_counter
from safetensors.torch import load_file

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import wandb

from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModel, AutoConfig
from transformers import DataCollatorForTokenClassification
from transformers import Trainer, TrainingArguments, TrainerCallback, EarlyStoppingCallback, set_seed
from transformers.modeling_outputs import TokenClassifierOutput
from seqeval.metrics import f1_score, precision_score, recall_score

ENVIRONMENT = "colab" if "google.colab" in sys.modules else "kaggle"

if ENVIRONMENT == "colab":
    from google.colab import userdata, drive
else:
    from kaggle_secrets import UserSecretsClient

In [5]:
root = Path("/content" if ENVIRONMENT == "colab" else "/kaggle/working")

repo = "UA-Locations-NER-Transformers"
branch = "main"
repo_dir = root / repo

if not repo_dir.exists():
    subprocess.run(["git", "clone", "-b", branch, f"https://github.com/dkalchenko01/{repo}.git", str(repo_dir)], check=True)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("Repository:", repo_dir)

Repository: /content/UA-Locations-NER-Transformers


Repository synchronization

In [23]:
subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
importlib.invalidate_caches()
for name in list(sys.modules):
    if name.startswith("src."):
        importlib.reload(sys.modules[name])

git_revision = subprocess.check_output(["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True).strip()
print("Git revision:", git_revision)

Git revision: 2a662ad8a79d0c3176d6a2cc4f91ad6561b5d4fc


Project imports

In [24]:
from src.data_io import get_data_dir, load_manifest, load_data_splits
from src.experiment_tracking import start_run, log_evaluation, finish_run, log_kaggle_result
from src.dataset import create_datasets, load_or_tokenize_datasets, prepare_labeling_dataset
from src.modeling import TransformerNER
from src.optimization import build_optimizer
from src.metrics import comp_metric
from src.preprocessing import bio_to_spans
from src.evaluation import build_compute_metrics
from src.callbacks import LabelingSampleCallback, CheckpointBackupCallback

splits_dir = repo_dir / "data" / "splits"

print("Modules imported successfully")
print("Validation manifest exists:", (splits_dir / "validation_manifest_v2.parquet").exists())

Modules imported successfully
Validation manifest exists: True


## External Services (W&B + Drive)

In [8]:
os.environ["WANDB_BASE_URL"] = "https://api.forge.coreweave.com"
os.environ["WANDB_DISABLE_CODE"] = "true"
os.environ["WANDB_CONSOLE"] = "off"

if ENVIRONMENT == "colab":
    wandb.login(key=userdata.get("WANDB_API_KEY"), relogin=True)
else:
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"), relogin=True)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: No netrc file found, creating one.
wandb: Appending key for api.forge.coreweave.com to your netrc file: /root/.netrc
wandb: Currently logged in as: daldoshyna (daldoshyna-kyiv-school-of-economics) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


In [9]:
if ENVIRONMENT == "colab":
    drive.mount("/content/drive")
    checkpoint_backup_root = Path("/content/drive/MyDrive/UA-Locations-NER-Transformers/xlm_roberta_large")
else:
    checkpoint_backup_root = root / "checkpoints/xlm_roberta_large"

checkpoint_backup_root.mkdir(parents=True, exist_ok=True)
print("Checkpoint backup root:", checkpoint_backup_root)

Mounted at /content/drive
Checkpoint backup root: /content/drive/MyDrive/UA-Locations-NER-Transformers/xlm_roberta_large


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Transformers:", version("transformers"))
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

PyTorch: 2.14.1+cu130
Transformers: 5.22.1
Device: cuda
GPU: NVIDIA A100-SXM4-80GB
BF16 supported: True


## Control Configuration

XLM-RoBERTa-large with learnable weighted pooling of the last 4 hidden layers and token-level BIO classification.

In [14]:
config = {
    "model": "xlm_roberta_large",
    "model_name": "FacebookAI/xlm-roberta-large",
    "architecture": "weighted_layer_pooling",
    "tokenization_version": "v1",
    "pooling_layers": 4,
    "dropout": 0.1,
    "label_to_id": {"O": 0, "B-LOC": 1, "I-LOC": 2},
    "seed": 17,
    "sources": ["uk", "ru"],
    "max_length": 512,
    "batch_size": 64,
    "eval_batch_size": 128,
    "gradient_accumulation_steps": 1,
    "trainable_encoder_layers": 8,
    "encoder_lr": 1e-5,
    "head_lr": 1e-4,
    "layer_lr_decay": 0.9,
    "optimizer": "adamw_torch",
    "weight_decay": 0.01,
    "scheduler": "cosine",
    "warmup_ratio": 0.1,
    "max_grad_norm": 1.0,
    "epochs": 3,
    "early_stopping_patience": 2,
    "environment": ENVIRONMENT,
    "validation_version": "v2",
    "manifest_name": "validation_manifest_v2",
    "metric_version": "kaggle_original_regex_v1",
    "wandb_entity": "daldoshyna-kyiv-school-of-economics",
    "wandb_project": "ua-location-ner-transformers",
    "decoding": "viterbi_lines",
    "postprocessing_version": "raw",
    "reannotated": True
}

set_seed(config["seed"])

Run directories

In [15]:
run_id = config["model"] + "_" + pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ")

run_dir = root / "outputs" / run_id
backup_dir = checkpoint_backup_root / run_id
manifest_path = splits_dir / f"{config['manifest_name']}.parquet"

run_dir.mkdir(parents=True, exist_ok=False)
backup_dir.mkdir(parents=True, exist_ok=True)

print("Run:", run_id)
print("Local directory:", run_dir)
print("Backup directory:", backup_dir)

Run: xlm_roberta_large_20261127T144947564069Z
Local directory: /content/outputs/xlm_roberta_large_20261127T144947564069Z
Backup directory: /content/drive/MyDrive/UA-Locations-NER-Transformers/xlm_roberta_large/xlm_roberta_large_20261127T144947564069Z


Provenance

In [16]:
provenance = {
    "run_id": run_id,
    "seed": config["seed"],
    "git_revision": git_revision,
    "git_branch": branch,
    "python": platform.python_version(),
    "packages": {name: version(name) for name in [
        "torch", "transformers", "tokenizers", "datasets",
        "accelerate", "numpy", "pandas", "pyarrow", "wandb"
    ]},
    "device": str(device),
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
}

## Data Loading

Reusing the fixed validation v2 splits from Assignment 3.

In [17]:
data_dir = get_data_dir(ENVIRONMENT)
manifest = load_manifest(splits_dir, name=config["manifest_name"])

display(pd.crosstab(manifest["source"], manifest["split"]))

100%|██████████| 811M/811M [00:46<00:00, 18.2MB/s]

Extracting files...


split,holdout,train,val
source,,,
ru,24743,100000,24626
uk,100757,805983,100741


Train, validation and holdout splits

In [18]:
splits = load_data_splits(
    ENVIRONMENT, splits_dir,
    sources=config["sources"],
    manifest_name=config["manifest_name"],
    reannotated=config["reannotated"]
)

for name, df in splits.items():
    print(f"{name}: {len(df):,} texts")

uk_train: 805,983 texts
uk_val: 100,741 texts
uk_holdout: 100,757 texts
ru_train: 100,000 texts
ru_val: 24,626 texts
ru_holdout: 24,743 texts


Test and labeling sample

In [19]:
test_df = pd.read_csv(data_dir / "test.csv")
labeling_sample = pd.read_csv(data_dir / "labeling_sample.csv")
print(f"Test: {len(test_df):,} texts")
print(f"Labeling sample: {len(labeling_sample):,} texts")
display(labeling_sample.head(3))

Test: 477 texts
Labeling sample: 26 texts


,text,locations,loc_markers
0,-261.310,[],[]
1,"Проспект Бандери стоїть, як і пів міста… довго...",['Проспект Бандери'],"[(0, 16)]"
2,☢️Радіаційний фон (мкР/год) по Київщині станом...,"['Київщині', 'Чорнобиль', 'Київ', 'Тетерів', '...","[(31, 39), (68, 77), (83, 87), (93, 100), (105..."


## Tokenizer Loading

Using the original pretrained XLM-RoBERTa tokenizer with character-level offset mapping for BIO label alignment.

Load pretrained tokenizer

In [20]:
tokenizer = AutoTokenizer.from_pretrained(config["model_name"], use_fast=True)

print("Tokenizer:", type(tokenizer).__name__)
print("Vocabulary size:", len(tokenizer))
print("Maximum sequence length:", tokenizer.model_max_length)
print("Fast tokenizer:", tokenizer.is_fast)
print("Special tokens:", tokenizer.special_tokens_map)

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Tokenizer: XLMRobertaTokenizer
Vocabulary size: 250002
Maximum sequence length: 512
Fast tokenizer: True
Special tokens: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}


## Dataset & Batching

Batched tokenization, BIO label alignment and dynamic padding using Hugging Face.

In [21]:
datasets = create_datasets(splits, config["sources"])
datasets

DatasetDict({
    train: Dataset({
        features: ['row_id', 'text', 'loc_markers', 'source', 'text_id'],
        num_rows: 905983
    })
    val: Dataset({
        features: ['row_id', 'text', 'loc_markers', 'source', 'text_id'],
        num_rows: 125367
    })
    holdout: Dataset({
        features: ['row_id', 'text', 'loc_markers', 'source', 'text_id'],
        num_rows: 125500
    })
})

Tokenization & Caching

In [22]:
cache_root = checkpoint_backup_root.parent / "tokenized_cache"
tokenizer_name = config["model_name"].split("/")[-1]
sources = "_".join(config["sources"])
annotation = "reannotated" if config["reannotated"] else "original"

cache_name = f"{tokenizer_name}_{sources}_{annotation}_{config['manifest_name']}_{config['max_length']}_{config['tokenization_version']}"
cache_path = cache_root / cache_name

tokenized_datasets = load_or_tokenize_datasets(datasets, tokenizer, config["label_to_id"], config["max_length"], cache_path)
tokenized_datasets

Map:   0%|          | 0/905983 [00:00<?, ? examples/s]

train: 1,219 alignment failures


Filter:   0%|          | 0/905983 [00:00<?, ? examples/s]

Map:   0%|          | 0/125367 [00:00<?, ? examples/s]

val: 0 alignment failures


Map:   0%|          | 0/125500 [00:00<?, ? examples/s]

holdout: 0 alignment failures


Saving the dataset (0/3 shards):   0%|          | 0/904764 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/125367 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/125500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['row_id', 'text', 'loc_markers', 'source', 'text_id', 'input_ids', 'attention_mask', 'offset_mapping', 'labels', 'valid'],
        num_rows: 904764
    })
    val: Dataset({
        features: ['row_id', 'text', 'loc_markers', 'source', 'text_id', 'input_ids', 'attention_mask', 'offset_mapping', 'labels', 'valid'],
        num_rows: 125367
    })
    holdout: Dataset({
        features: ['row_id', 'text', 'loc_markers', 'source', 'text_id', 'input_ids', 'attention_mask', 'offset_mapping', 'labels', 'valid'],
        num_rows: 125500
    })
})

Dynamic padding

In [25]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True, label_pad_token_id=-100, pad_to_multiple_of=8)

Labeling sample tokenization

In [26]:
tokenized_labeling = prepare_labeling_dataset(labeling_sample, tokenizer, config["label_to_id"], config["max_length"])

Map:   0%|          | 0/26 [00:00<?, ? examples/s]

## Model Architecture

Pretrained XLM-RoBERTa encoder with learnable weighted layer pooling, dropout and a token-level BIO classification head.

In [27]:
model = TransformerNER(
    model_name=config["model_name"],
    num_labels=len(config["label_to_id"]),
    pooling_layers=config["pooling_layers"],
    dropout=config["dropout"]
).to(device)
model.freeze_encoder(config["trainable_encoder_layers"])

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [28]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable share: {trainable_params / total_params:.2%}")

Total parameters: 559,893,511
Trainable parameters: 100,772,871
Trainable share: 18.00%


## Training

### Training Arguments

In [31]:
training_args = TrainingArguments(
    output_dir=str(run_dir / "checkpoints"),
    num_train_epochs=config["epochs"],
    per_device_train_batch_size=config["batch_size"],
    per_device_eval_batch_size=config["eval_batch_size"],
    gradient_accumulation_steps=config["gradient_accumulation_steps"],
    learning_rate=config["encoder_lr"],
    weight_decay=config["weight_decay"],
    optim=config["optimizer"],
    lr_scheduler_type=config["scheduler"],
    warmup_steps=config["warmup_ratio"],
    max_grad_norm=config["max_grad_norm"],
    eval_strategy="epoch",
    save_strategy="epoch",
    train_sampling_strategy="group_by_length",
    batch_eval_metrics=True,
    include_for_metrics=["inputs"],
    logging_strategy="steps",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="uk_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="wandb",
    run_name=run_id,
    seed=config["seed"],
    remove_unused_columns=True,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported()
)

### Evaluation Metrics

In [32]:
compute_metrics = build_compute_metrics(tokenized_datasets["val"], config["label_to_id"])

### Trainer Initialization

In [33]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["val"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stopping_patience"])]
)

### Optimizer Initialization

In [34]:
trainer.optimizer = build_optimizer(trainer, config)

### Callbacks

In [35]:
trainer.add_callback(LabelingSampleCallback(trainer, tokenized_labeling, config["label_to_id"]))
trainer.add_callback(CheckpointBackupCallback(backup_dir))

### W&B Run Initialization

In [36]:
run = start_run(
    model=config["model"],
    config=config,
    name=run_id,
    validation_version=config["validation_version"],
    metric_version=config["metric_version"],
    postprocessing_version=config["postprocessing_version"],
    entity=config["wandb_entity"],
    project=config["wandb_project"],
    repo_dir=repo_dir,
    split_manifest_path=manifest_path,
    provenance=provenance
)

### Model Training

In [37]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss


ValueError: Evaluation dataset order mismatch

Urgent fixes

In [38]:
recovery_dir = backup_dir / "recovery_epoch_1"
trainer.save_model(str(recovery_dir))
trainer.state.save_to_json(str(recovery_dir / "trainer_state.json"))
print("Saved:", recovery_dir)

Saved: /content/drive/MyDrive/UA-Locations-NER-Transformers/xlm_roberta_large/xlm_roberta_large_20261127T144947564069Z/recovery_epoch_1


In [39]:
torch.save(trainer.optimizer.state_dict(), recovery_dir / "optimizer.pt")
torch.save(trainer.lr_scheduler.state_dict(), recovery_dir / "scheduler.pt")
print([(p.name, round(p.stat().st_size / 1024**2, 2)) for p in recovery_dir.iterdir()])

[('model.safetensors', 2135.87), ('tokenizer_config.json', 0.0), ('tokenizer.json', 16.31), ('training_args.bin', 0.01), ('trainer_state.json', 0.03), ('optimizer.pt', 768.94), ('scheduler.pt', 0.0)]


In [42]:
from torch.utils.data import SequentialSampler
from types import MethodType

def get_sequential_eval_sampler(self, eval_dataset):
    return SequentialSampler(eval_dataset)

trainer._get_eval_sampler = MethodType(get_sequential_eval_sampler, trainer)
trainer.args.train_sampling_strategy = "group_by_length"

In [43]:
batch = next(iter(trainer.get_eval_dataloader()))

for i in range(5):
    actual_ids = batch["input_ids"][i][batch["attention_mask"][i].bool()].tolist()
    expected_ids = tokenized_datasets["val"][i]["input_ids"]
    print(i, actual_ids == expected_ids)

0 True
1 True
2 True
3 True
4 True


Continue training

In [45]:
trainer.compute_metrics = build_compute_metrics(tokenized_datasets["val"], config["label_to_id"])

trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=config["early_stopping_patience"]))
trainer.add_callback(LabelingSampleCallback(trainer, tokenized_labeling, config["label_to_id"]))

trainer.args.ignore_data_skip = False
trainer.state.save_to_json(str(recovery_dir / "trainer_state.json"))

train_result = trainer.train(resume_from_checkpoint=str(recovery_dir))

Epoch,Training Loss,Validation Loss,Entity F1,Entity Precision,Entity Recall,F1,Precision,Recall,Tp,Fp,Fn,Ru F1,Ru Precision,Ru Recall,Ru Tp,Ru Fp,Ru Fn,Uk F1,Uk Precision,Uk Recall,Uk Tp,Uk Fp,Uk Fn
2,0.018612,0.024056,0.845447,0.852060,0.838936,0.848186,0.855174,0.841310,42503,7198,8017,0.816216,0.823329,0.809225,9684,2078,2283,0.858103,0.865047,0.851270,32819,5120,5734
3,0.017875,0.023752,0.850631,0.844155,0.857207,0.853479,0.847540,0.859501,43422,7811,7098,0.820189,0.816151,0.824267,9864,2222,2103,0.863784,0.857230,0.870438,33558,5589,4995


Checkpoint backed up: /content/drive/MyDrive/UA-Locations-NER-Transformers/xlm_roberta_large/xlm_roberta_large_20261127T144947564069Z/checkpoint-28274
Checkpoint backed up: /content/drive/MyDrive/UA-Locations-NER-Transformers/xlm_roberta_large/xlm_roberta_large_20261127T144947564069Z/checkpoint-42411


## Checkpoint Selection

In [46]:
checkpoints = sorted(backup_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))

for checkpoint in checkpoints:
    print(checkpoint.name)

print("Automatic best:", Path(trainer.state.best_model_checkpoint).name)
print("Best UK F1:", trainer.state.best_metric)

checkpoint-28274
checkpoint-42411
Automatic best: checkpoint-42411
Best UK F1: 0.86378378


In [47]:
pd.DataFrame([
    log for log in trainer.state.log_history
    if "labeling_f1" in log
])[["epoch", "step", "labeling_f1", "labeling_precision", "labeling_recall", "labeling_TP", "labeling_FP", "labeling_FN"]]

,epoch,step,labeling_f1,labeling_precision,labeling_recall,labeling_TP,labeling_FP,labeling_FN
0,2.0,28274,0.666667,0.527273,0.90625,29,26,3
1,3.0,42411,0.651685,0.508772,0.90625,29,28,3


In [48]:
selected_checkpoint = None

automatic_best = Path(trainer.state.best_model_checkpoint).name
selected_checkpoint = selected_checkpoint or automatic_best
selected_checkpoint_path = backup_dir / selected_checkpoint

print("Selected checkpoint:", selected_checkpoint_path)

Selected checkpoint: /content/drive/MyDrive/UA-Locations-NER-Transformers/xlm_roberta_large/xlm_roberta_large_20261127T144947564069Z/checkpoint-42411


In [60]:
selected_checkpoint = "checkpoint-28274"
selected_checkpoint_path = backup_dir / selected_checkpoint
print("Selected checkpoint:", selected_checkpoint_path)

Selected checkpoint: /content/drive/MyDrive/UA-Locations-NER-Transformers/xlm_roberta_large/xlm_roberta_large_20261127T144947564069Z/checkpoint-28274


### Load Selected Checkpoint

In [61]:
weights = load_file(str(selected_checkpoint_path / "model.safetensors"))
trainer.model.load_state_dict(weights)
trainer.model.eval()
del weights

## Final Evaluation

### Holdout Evaluation

In [50]:
trainer.remove_callback(LabelingSampleCallback)
trainer.remove_callback(EarlyStoppingCallback)
trainer.compute_metrics = build_compute_metrics(tokenized_datasets["holdout"], config["label_to_id"])
holdout_metrics = trainer.evaluate(tokenized_datasets["holdout"], metric_key_prefix="holdout")

Training Loss,Validation Loss,Epoch,Entity F1,Entity Precision,Entity Recall,F1,Precision,Recall,Tp,Fp,Fn,Ru F1,Ru Precision,Ru Recall,Ru Tp,Ru Fp,Ru Fn,Uk F1,Uk Precision,Uk Recall,Uk Tp,Uk Fp,Uk Fn
0.017875,0.024064,3,0.850724,0.844058,0.857496,0.853495,0.847301,0.859779,43847,7902,7151,0.817396,0.807842,0.827179,9745,2318,2036,0.864403,0.859295,0.869572,34102,5584,5115


### Labeling Sample Evaluation

In [51]:
trainer.compute_metrics = build_compute_metrics(tokenized_labeling, config["label_to_id"])
labeling_metrics = trainer.evaluate(tokenized_labeling, metric_key_prefix="labeling")

Training Loss,Validation Loss,Epoch,Entity F1,Entity Precision,Entity Recall,F1,Precision,Recall,Tp,Fp,Fn
0.017875,0.169306,3,0.637363,0.491525,0.906250,0.651685,0.508772,0.906250,29,28,3


## Saving Results

In [52]:
results = {
    "automatic_best_checkpoint": automatic_best,
    "best_uk_f1": trainer.state.best_metric,
    "selected_checkpoint": selected_checkpoint,
    "holdout": holdout_metrics,
    "labeling": labeling_metrics
}

for name, content in [("config", config), ("provenance", provenance), ("metrics", results), ("training_history", trainer.state.log_history)]:
    with open(backup_dir / f"{name}.json", "w") as f:
        json.dump(content, f, indent=4, ensure_ascii=False, default=str)

## Inference & Post-processing

### Test Tokenization

In [53]:
test_encodings = tokenizer(test_df["text"].tolist(), truncation=True, max_length=config["max_length"], return_offsets_mapping=True)
test_dataset = Dataset.from_dict(dict(test_encodings))

### Test Predictions

In [62]:
from src.evaluation import decode_batch
from src.decoding import decode_spans

id_to_label = {v: k for k, v in config["label_to_id"].items()}
test_spans = []
test_logits = []

trainer.model.eval()

with torch.inference_mode():
    for start in range(0, len(test_dataset), config["eval_batch_size"]):
        rows = test_dataset.select(range(start, min(start + config["eval_batch_size"], len(test_dataset)))).to_list()
        features = [{k: v for k, v in row.items() if k != "offset_mapping"} for row in rows]
        batch = data_collator(features).to(device)

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = trainer.model(**batch).logits.float().cpu()

        records = [{**row, "text": test_df.iloc[start + i]["text"]} for i, row in enumerate(rows)]
        predictions, indices = decode_batch(logits, records, config["label_to_id"])

        for i, (row, pred, idx) in enumerate(zip(rows, predictions, indices)):
            offsets = [row["offset_mapping"][j] for j in idx]
            test_spans.append(decode_spans(pred, offsets, id_to_label))
            test_logits.append(logits[i, :len(row["input_ids"])].clone())

In [63]:
torch.save({"logits": test_logits, "offsets": test_encodings["offset_mapping"], "spans": test_spans}, backup_dir / f"{selected_checkpoint}_test_predictions.pt")

### Post-processing

In [64]:
from src.postprocessing import postprocess, VARIANTS

postprocessing_version = "v5"

test_locations = [
    postprocess(text, spans, **VARIANTS[postprocessing_version])
    for text, spans in zip(test_df["text"], test_spans)
]

### Submission

In [65]:
submission = test_df[["text_id"]].copy()
submission["locations"] = [str(locations) for locations in test_locations]

submission_path = backup_dir / f"submission_{selected_checkpoint}_{postprocessing_version}.csv"
submission.to_csv(submission_path, index=False)

print(submission.shape)
display(submission.head())

(477, 2)


,text_id,locations
0,0,['Київ']
1,1,"['Києві', 'Києва', 'Шулявського шляхопроводу']"
2,2,"['селищу Гоголеве', 'Миргородського району', '..."
3,3,[]
4,4,"['Льодовий стадіон', 'проспекті Академіка Глуш..."


In [66]:
from google.colab import files

files.download(str(submission_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Sliding Window

In [70]:
selected_checkpoint = "checkpoint-42411"
selected_checkpoint_path = backup_dir / selected_checkpoint

weights = load_file(str(selected_checkpoint_path / "model.safetensors"))
trainer.model.load_state_dict(weights)
trainer.model.eval()
del weights

In [67]:
full_encodings = tokenizer(test_df["text"].tolist(), truncation=False, verbose=False)
long_indices = [i for i, ids in enumerate(full_encodings["input_ids"]) if len(ids) > config["max_length"]]

print("Long texts:", len(long_indices))

Long texts: 8


In [68]:
text = test_df.iloc[long_indices[0]]["text"]

windows = tokenizer(
    text,
    truncation=True,
    max_length=config["max_length"],
    stride=128,
    return_overflowing_tokens=True,
    return_offsets_mapping=True
)

print("Windows:", len(windows["input_ids"]))
print("Window lengths:", [len(ids) for ids in windows["input_ids"]])

Windows: 2
Window lengths: [512, 347]


In [69]:
from collections import defaultdict
from src.decoding import viterbi_decode, decode_spans

def predict_sliding_window(text, stride=128):
    windows = tokenizer(text, truncation=True, max_length=config["max_length"], stride=stride, return_overflowing_tokens=True, return_offsets_mapping=True)
    token_logits = defaultdict(list)

    with torch.inference_mode():
        for i, offsets in enumerate(windows["offset_mapping"]):
            features = {k: torch.tensor([windows[k][i]], device=device) for k in ["input_ids", "attention_mask"]}

            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                logits = trainer.model(**features).logits[0].float().cpu()

            for offset, logit in zip(offsets, logits):
                if offset[0] < offset[1]:
                    token_logits[tuple(offset)].append(logit)

    offsets = sorted(token_logits)
    logits = torch.stack([torch.stack(token_logits[offset]).mean(dim=0) for offset in offsets])
    predictions = viterbi_decode(logits.unsqueeze(0), [len(offsets)], config["label_to_id"], [{"text": text, "offsets": offsets}])[0]
    spans = decode_spans(predictions, offsets, id_to_label)

    return spans, logits, offsets

In [71]:
saved = torch.load(backup_dir / f"{selected_checkpoint}_test_predictions.pt", weights_only=True)
sliding_spans = saved["spans"].copy()
sliding_logits = saved["logits"].copy()
sliding_offsets = saved["offsets"].copy()

for i in long_indices:
    sliding_spans[i], sliding_logits[i], sliding_offsets[i] = predict_sliding_window(test_df.iloc[i]["text"])

torch.save({"spans": sliding_spans, "logits": sliding_logits, "offsets": sliding_offsets}, backup_dir / f"{selected_checkpoint}_sliding_predictions.pt")

In [72]:
submission_sliding = test_df[["text_id"]].copy()
submission_sliding["locations"] = [str(postprocess(text, spans, **VARIANTS["v5"])) for text, spans in zip(test_df["text"], sliding_spans)]

sliding_path = backup_dir / f"submission_{selected_checkpoint}_sliding_v5.csv"
submission_sliding.to_csv(sliding_path, index=False)

files.download(str(sliding_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Threshold Tuning

In [73]:
def collect_predictions(dataset):
    all_spans, all_logits, all_offsets = [], [], []
    trainer.model.eval()

    with torch.inference_mode():
        for start in range(0, len(dataset), config["eval_batch_size"]):
            rows = dataset.select(range(start, min(start + config["eval_batch_size"], len(dataset)))).to_list()
            features = [{k: row[k] for k in ["input_ids", "attention_mask"]} for row in rows]
            batch = data_collator(features).to(device)

            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                logits = trainer.model(**batch).logits.float().cpu()

            predictions, indices = decode_batch(logits, rows, config["label_to_id"])

            for i, (row, pred, idx) in enumerate(zip(rows, predictions, indices)):
                offsets = [row["offset_mapping"][j] for j in idx]
                all_spans.append(decode_spans(pred, offsets, id_to_label))
                all_logits.append(logits[i, :len(row["input_ids"])].clone())
                all_offsets.append(row["offset_mapping"])

    return {"spans": all_spans, "logits": all_logits, "offsets": all_offsets}

In [74]:
labeling_predictions = collect_predictions(tokenized_labeling)
torch.save(labeling_predictions, backup_dir / f"{selected_checkpoint}_labeling_predictions.pt")

holdout_predictions = collect_predictions(tokenized_datasets["holdout"])
torch.save(holdout_predictions, backup_dir / f"{selected_checkpoint}_holdout_predictions.pt")

print("Predictions saved")

Predictions saved


### Finish W&B Run

In [75]:
finish_run(
    run,
    summary={**holdout_metrics, **labeling_metrics, "selected_checkpoint": selected_checkpoint},
    best_score=trainer.state.best_metric,
    selection_metric="uk_f1",
    checkpoint_path=backup_dir / automatic_best,
    upload_checkpoint=False
)

eval/FN,▁█▇
eval/FP,▁▇█
eval/TP,▁██
eval/entity_f1,▁▃█
eval/entity_precision,▁█▆
eval/entity_recall,█▁█
eval/f1,▁▅█
eval/loss,▁██
eval/precision,▁█▆
eval/recall,▆▁█
+58,...
